In [1]:
import cv2
import numpy as np

# 1. Open the webcam
cap = cv2.VideoCapture(0)

# Read the first frame and convert to Grayscale
ret, frame1 = cap.read()
prvs_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# Create a blank image to hold our HSV color map
# HSV stands for Hue (Color), Saturation (Intensity), Value (Brightness)
hsv_mask = np.zeros_like(frame1)
hsv_mask[..., 1] = 255 # Set maximum Saturation so colors pop

print("Dense Optical Flow Active. Press 'q' to quit.")

while True:
    ret, frame2 = cap.read()
    if not ret:
        break
        
    next_gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    
    # 2. Calculate Dense Optical Flow (Farneback Algorithm)
    # This returns a 2D array of vectors: [dx, dy] for every single pixel
    flow = cv2.calcOpticalFlowFarneback(prvs_gray, next_gray, None, 
                                        pyr_scale=0.5, levels=3, winsize=15, 
                                        iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
    
    # 3. Convert the vectors (x, y) into Angles and Speed (Magnitude)
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    
    # 4. Map the math to colors!
    # Angle determines the Hue (which color on the rainbow)
    hsv_mask[..., 0] = angle * 180 / np.pi / 2 
    # Speed determines the Brightness (faster movement = brighter color)
    hsv_mask[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    
    # Convert our HSV map back to standard BGR colors so the screen can display it
    rgb_flow = cv2.cvtColor(hsv_mask, cv2.COLOR_HSV2BGR)
    
    # Show the result
    cv2.imshow('Optical Flow (Color = Direction, Brightness = Speed)', rgb_flow)
    cv2.imshow('Original Camera', frame2)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
    # Update the previous frame to the current frame for the next loop
    prvs_gray = next_gray

cap.release()
cv2.destroyAllWindows()

Dense Optical Flow Active. Press 'q' to quit.
